In [11]:
import pandas as pd
import numpy as np

# 1. CARICAMENTO E PREPARAZIONE DATI
df = pd.read_csv("../data/raw/demand.csv")

# Pulizia Order_Demand: sostituzione parentesi con segno meno e conversione a numerico[cite: 1, 2]
df["Order_Demand"] = (
    df["Order_Demand"]
    .astype(str)
    .str.strip()
    .str.replace("(", "-", regex=False)
    .str.replace(")", "", regex=False)
)
df["Order_Demand"] = pd.to_numeric(df["Order_Demand"])

# Pulizia Date: conversione e filtro per il periodo affidabile 2012-2016[cite: 2]
df["Date"] = pd.to_datetime(df["Date"], format="%Y/%m/%d")
df = df.dropna(subset=["Date"])
df_analisi = df[(df["Date"] >= "2012-01-01") & (df["Date"] <= "2016-12-31")].copy()

In [12]:
# 2. CALCOLO DOMANDA MEDIA GIORNALIERA REALE
# Sommiamo tutta la domanda nel periodo per singolo prodotto
domanda_totale = df_analisi.groupby("Product_Code")["Order_Demand"].sum().reset_index()

In [13]:
# Calcoliamo i giorni totali del periodo (dal 2012-01-01 al 2016-12-31 sono 1827 giorni)[cite: 2]
giorni_totali = (pd.to_datetime("2016-12-31") - pd.to_datetime("2012-01-01")).days + 1


In [14]:
# Calcoliamo la media giornaliera
domanda_totale["Avg_Daily_Demand"] = (domanda_totale["Order_Demand"] / giorni_totali).round(1)

In [15]:
# Teniamo solo i prodotti con una domanda media positiva per le nostre regole di riordino
df_business = domanda_totale[domanda_totale["Avg_Daily_Demand"] > 0].copy()

In [16]:
# 3. SIMULAZIONE DATI DI MAGAZZINO
np.random.seed(42)
n_prodotti = len(df_business)
# Simuliamo uno stock coerente con i volumi di vendita
df_business['Current_Stock'] = (df_business['Avg_Daily_Demand'] * np.random.uniform(10, 60, size=n_prodotti)).astype(int)
df_business['Lead_Time_Days'] = np.random.choice([7, 14, 21, 30], size=n_prodotti)
df_business['Min_Order_Qty'] = np.random.choice([50, 100, 500, 1000], size=n_prodotti)

# 4. APPLICAZIONE REGOLE DI BUSINESS
# A. Scorta di Sicurezza (Copre il 50% della domanda durante il Lead Time)
df_business['Safety_Stock'] = (df_business['Avg_Daily_Demand'] * df_business['Lead_Time_Days'] * 0.5).astype(int)

# B. Punto di Riordino (Domanda nel Lead Time + Scorta di Sicurezza)
df_business['Reorder_Point'] = (df_business['Avg_Daily_Demand'] * df_business['Lead_Time_Days']).astype(int) + df_business['Safety_Stock']

# C. Monitoraggio Scorte e Autonomia
df_business['Stock_Operativo'] = df_business['Current_Stock'] - df_business['Safety_Stock']
df_business['Days_to_Safety_Stock'] = (df_business['Stock_Operativo'] / df_business['Avg_Daily_Demand']).round(1)
df_business['Total_Days_to_Zero'] = (df_business['Current_Stock'] / df_business['Avg_Daily_Demand']).round(1)

# D. Calcolo Quantità d'Ordine
df_business['Needs_Reorder'] = df_business['Current_Stock'] <= df_business['Reorder_Point']
quantita_teorica = df_business['Reorder_Point'] - df_business['Current_Stock']

df_business['Order_Quantity'] = np.where(
    df_business['Needs_Reorder'],
    np.maximum(quantita_teorica, df_business['Min_Order_Qty']),
    0
).astype(int)

# E. Semafori di Urgenza
def calcola_urgenza(row):
    if not row['Needs_Reorder']:
        return "🟢 OK"
    if row['Total_Days_to_Zero'] <= row['Lead_Time_Days']:
        return "🔴 CRITICO: Stock-Out"
    elif row['Days_to_Safety_Stock'] <= 0:
        return "🟠 ALTO: Intaccata Sicurezza"
    else:
        return "🟡 NORMALE: Da Ordinare"

df_business['Urgency'] = df_business.apply(calcola_urgenza, axis=1)

In [17]:
# 5. OUTPUT CRUSCOTTO /content/Historical Product Demand .csv
colonne_finali = [
    'Product_Code', 'Current_Stock', 'Safety_Stock', 'Reorder_Point',
    'Lead_Time_Days', 'Total_Days_to_Zero', 'Urgency', 'Order_Quantity'
]
df_final = df_business[colonne_finali].sort_values('Total_Days_to_Zero')

print("--- TOP 10 PRODOTTI CRITICI DA ORDINARE ---")
display(df_final.head(10))

--- TOP 10 PRODOTTI CRITICI DA ORDINARE ---


,Product_Code,Current_Stock,Safety_Stock,Reorder_Point,Lead_Time_Days,Total_Days_to_Zero,Urgency,Order_Quantity
1087,Product_1098,8,2,7,7,10.0,🟢 OK,0
482,Product_0483,2,0,1,7,10.0,🟢 OK,0
499,Product_0500,1,0,0,7,10.0,🟢 OK,0
2048,Product_2061,3,4,13,30,10.0,🔴 CRITICO: Stock-Out,500
1928,Product_1940,3,3,9,21,10.0,🔴 CRITICO: Stock-Out,100
1935,Product_1947,1,1,4,30,10.0,🔴 CRITICO: Stock-Out,500
948,Product_0959,1,1,3,21,10.0,🔴 CRITICO: Stock-Out,50
933,Product_0944,1,0,1,14,10.0,🔴 CRITICO: Stock-Out,100
1763,Product_1774,15,10,31,14,10.0,🔴 CRITICO: Stock-Out,100
806,Product_0807,2,3,9,30,10.0,🔴 CRITICO: Stock-Out,50


# 03 - Regole di business: quanto e quando ordinare

In questo notebook trasformiamo la domanda storica in una **raccomandazione d'ordine** per ogni prodotto:
serve riordinare? quanto? con quale urgenza?

## Cosa fa il codice

1. **Carica e pulisce i dati** come nei notebook precedenti: `Order_Demand` diventa un numero
   (i valori tra parentesi diventano negativi) e `Date` una data vera. Si usa solo il periodo
   affidabile, 2012-2016 (1.827 giorni).
2. **Calcola la domanda media giornaliera** di ogni prodotto: domanda totale / 1.827 giorni.
   Restano solo i prodotti con domanda media positiva.
3. **Simula i dati che il dataset non contiene**, con numeri casuali (`seed` fisso, quindi ripetibili):
   - stock attuale: domanda giornaliera × un numero casuale tra 10 e 60
   - lead time (giorni di consegna del fornitore): 7, 14, 21 o 30
   - ordine minimo del fornitore: 50, 100, 500 o 1000
4. **Applica le regole di riordino:**

| Calcolo | Formula |
|---|---|
| Scorta di sicurezza | domanda giornaliera × lead time × 0,5 |
| Punto di riordino | domanda giornaliera × lead time + scorta di sicurezza |
| Giorni alla scorta di sicurezza | (stock − scorta di sicurezza) / domanda giornaliera |
| Giorni a zero | stock / domanda giornaliera |
| Serve riordinare? | stock ≤ punto di riordino |
| Quantità da ordinare | il maggiore tra (punto di riordino − stock) e l'ordine minimo; 0 se non serve |

5. **Assegna un semaforo di urgenza**, valutando in questo ordine:
   - 🟢 **OK**: stock sopra il punto di riordino
   - 🔴 **CRITICO**: i giorni a zero sono ≤ al lead time (lo stock finisce prima che arrivi la merce)
   - 🟠 **ALTO**: la scorta di sicurezza è già intaccata
   - 🟡 **NORMALE**: da ordinare, ma con margine
6. **Mostra i 10 prodotti più vicini a finire**, ordinati per giorni a zero.

## Come leggere la tabella finale

- `Reorder_Point`: livello di stock sotto cui bisogna ordinare.
- `Total_Days_to_Zero`: tra quanti giorni finisce lo stock, alla velocità di vendita media.
- `Order_Quantity`: quanti pezzi ordinare (0 se il prodotto è in verde).

## Limiti (da tenere presenti)

- **Stock, lead time e ordine minimo sono simulati.** I numeri della tabella dimostrano il meccanismo,
  non sono risultati reali. Poiché lo stock simulato vale almeno 10 giorni di domanda, i prodotti
  in cima alla tabella risultano tutti a circa 10 giorni: dipende dalla simulazione, non dai prodotti.
- **L'ordine minimo è casuale** e non dipende dal prodotto: per prodotti che vendono pochissimo può
  portare a quantità molto grandi rispetto alla domanda.
- **La scorta di sicurezza è un'euristica** (50% della domanda nel lead time) e non tiene conto
  di quanto la domanda varia.
- **La domanda è la media storica**, non una previsione: nel progetto completo la sostituiremo
  con l'output del modello di previsione.
- **Non c'è il portafoglio ordini** (merce già ordinata e in arrivo) e il calcolo è per prodotto,
  con tutti i magazzini insieme.
- Per i prodotti con domanda molto bassa l'arrotondamento a un decimale e il troncamento a intero
  rendono i valori meno precisi.